# MediShield AI: Explainable AI Framework for Early Prediction of 30-Day Patient Readmission Risk

**Healthcare Analytics Mini Project**  
**Dataset:** Diabetes 130-US Hospitals for Years 1999–2008  
**Primary task:** Binary classification of 30-day readmission (`<30` vs not `<30`)

### Project objective
Develop and evaluate an **explainable, imbalance-aware machine-learning framework** that estimates a diabetic patient's risk of readmission within 30 days using information available during the hospitalization. The notebook combines healthcare-specific feature engineering, leakage-aware preprocessing, four complementary ML algorithms, validation-based model selection, threshold optimization, an ensemble, SHAP explanations, risk stratification, and healthcare insights.

### What makes MediShield AI different from a basic ML comparison
1. **Healthcare-specific feature engineering:** diagnosis groups, medication-regimen summaries, prior-utilization burden, and clinically meaningful derived variables.
2. **Leakage-aware patient handling:** identifiers are removed and only one encounter per patient is retained for the primary modeling cohort.
3. **Imbalance-aware learning:** SMOTE is applied **only to the training partition**; validation and test data remain untouched.
4. **Multi-metric model selection:** selection is not driven by accuracy alone because the 30-day readmission class is highly imbalanced.
5. **Validation-only threshold optimization:** the operating threshold is selected on validation data and applied once to the untouched test set.
6. **Explainable ensemble:** SHAP is used to explain global and individual risk predictions.

> **Important:** All performance values in this notebook are generated from the supplied dataset. No result is hard-coded or copied from an external source.

## 1. Reproducible setup and input discovery

The notebook first looks for the official diabetes CSV in the working directory and common upload locations. If the Kaggle/Moodle copy has a different filename, set `DATA_PATH` manually in the next cell.

The project also creates folders for cleaned data, figures, metrics, models, and dashboard-ready outputs.

In [ ]:
import os, re, json, time, warnings, glob, joblib
from pathlib import Path
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

RANDOM_STATE = 42
PROJECT_DIR = Path("./MediShield_AI_Project")
DATA_DIR = PROJECT_DIR / "data"
FIG_DIR = PROJECT_DIR / "figures"
OUT_DIR = PROJECT_DIR / "outputs"
MODEL_DIR = PROJECT_DIR / "models"
for d in [DATA_DIR, FIG_DIR, OUT_DIR, MODEL_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# Automatic discovery. Override DATA_PATH if necessary.
DATA_PATH = None
candidates = []
for root in [Path("."), Path("/mnt/data"), Path("/mnt/user-data/uploads"), Path("/content")]:
    if root.exists():
        candidates += list(root.glob("**/*diabetic_data*.csv"))
        candidates += list(root.glob("**/*Diabetes*130*.csv"))

if DATA_PATH is None and candidates:
    DATA_PATH = str(candidates[0])

print("Detected DATA_PATH:", DATA_PATH)
print("Project directory:", PROJECT_DIR.resolve())
if DATA_PATH is None:
    print("Set DATA_PATH manually to diabetic_data.csv before running the loading cell.")

# MediShield AI — Proposed Model and Framework

## What is MediShield AI?

**MediShield AI is the proposed explainable ensemble-based machine-learning framework developed in this project. It is not a standalone conventional machine-learning algorithm such as Logistic Regression, Random Forest, XGBoost, Gradient Boosting, or LightGBM.**

The individual machine-learning algorithms are first evaluated as **candidate models**. Their validation performance is compared using healthcare-relevant classification metrics. The strongest candidate models are then combined through a **weighted ensemble** to form the final predictive component of MediShield AI.

The complete MediShield AI framework integrates:

1. Healthcare-specific feature engineering
2. Patient-level data handling to reduce information leakage
3. Class-imbalance handling using the training data only
4. Multiple candidate machine-learning models
5. Validation-based selection of strong candidate models
6. Weighted ensemble prediction
7. Validation-based decision-threshold optimization for 30-day readmission detection
8. SHAP-based explainable AI for global and individual predictions
9. Patient risk stratification and healthcare interpretation

Therefore, **MediShield AI represents the proposed methodology/framework of this project**, while the conventional algorithms are used as baselines and candidate components within that framework.

### Model Roles in the Project

| Model | Role in the project |
|---|---|
| Logistic Regression | Baseline interpretable classification model |
| Random Forest | Candidate machine-learning model |
| XGBoost | Candidate boosting model |
| Gradient Boosting | Candidate boosting model |
| LightGBM | Candidate boosting model |
| Other evaluated models | Candidate models for validation comparison |
| **MediShield AI** | **Proposed weighted ensemble-based explainable framework** |

### MediShield AI Development Flow

```text
Patient Encounter Data
        ↓
Healthcare-Specific Preprocessing & Feature Engineering
        ↓
Class-Imbalance Handling on Training Data
        ↓
Candidate ML Models
        ↓
Validation-Based Multi-Metric Comparison
        ↓
Selection of Strong Candidate Models
        ↓
Weighted Ensemble
        ↓
MediShield AI Predictive Model
        ↓
Validation-Based Threshold Optimization
        ↓
30-Day Readmission Risk
        ↓
SHAP Explainability
        ↓
Global & Patient-Level Healthcare Insights
```

### Why this is the proposed approach

A standard approach would stop after training and comparing individual algorithms. In this project, the proposed approach goes further by integrating **healthcare-specific feature engineering, imbalance-aware learning, ensemble prediction, threshold optimization and explainability** into one readmission-risk framework. This integrated framework is referred to as **MediShield AI** throughout the project.

## 2. Dataset loading and data audit

The modeling target is derived from the original `readmitted` field:

- `<30` → **1: early readmission within 30 days**
- `>30` → **0: no early readmission**
- `NO` → **0: no early readmission**

This converts the original three-category readmission field into the project's specific 30-day-risk question.

In [ ]:
if DATA_PATH is None:
    raise FileNotFoundError("Set DATA_PATH to the Moodle/Kaggle-provided diabetic_data.csv before running this cell.")

df_raw = pd.read_csv(DATA_PATH)
print("Raw dataset shape:", df_raw.shape)
print("Columns:", len(df_raw.columns))
display(df_raw.head())

assert "readmitted" in df_raw.columns, "The supplied file must contain the readmitted column."
print("\nOriginal target distribution:")
display(df_raw["readmitted"].value_counts(dropna=False).rename_axis("readmitted").reset_index(name="count"))
print("\nMissing-value sentinel counts:")
display((df_raw == "?").sum().sort_values(ascending=False).head(15).to_frame("question_mark_count"))

In [ ]:
# Dataset audit table
profile = pd.DataFrame({
    "dtype": df_raw.dtypes.astype(str),
    "missing_count": df_raw.isna().sum(),
    "question_mark_count": (df_raw == "?").sum(),
    "unique_values": df_raw.nunique(dropna=False)
}).sort_values("missing_count", ascending=False)
display(profile.head(20))

plt.figure(figsize=(7,4))
sns.countplot(data=df_raw, x="readmitted", order=["NO", ">30", "<30"])
plt.title("Original Readmission Distribution")
plt.xlabel("Readmission category")
plt.ylabel("Encounters")
plt.tight_layout()
plt.savefig(FIG_DIR / "01_original_target_distribution.png", dpi=180, bbox_inches="tight")
plt.show()

## 3. Data preprocessing and leakage control

### Decisions used in MediShield AI
- Replace `?` with missing values.
- Remove death/hospice discharge encounters because they cannot subsequently be readmitted in the dataset's target definition.
- Retain only the first encounter per patient for the primary cohort to avoid the same patient appearing repeatedly in the modeling population.
- Remove direct identifiers (`encounter_id`, `patient_nbr`).
- Drop extremely sparse fields such as `weight` and `payer_code` rather than creating a large amount of artificial information.
- Bucket rare medical specialties while retaining the most frequent specialties.
- Preserve informative missingness for laboratory tests by using `NotTested` as a category.

The exact row counts are recorded rather than assumed.

In [ ]:
df = df_raw.copy().replace("?", np.nan)
pre_log = {"raw_rows": int(len(df)), "raw_columns": int(df.shape[1])}

# Exclude encounters ending in death/hospice because they cannot contribute to subsequent readmission.
death_hospice_codes = [11, 13, 14, 19, 20, 21]
if "discharge_disposition_id" in df.columns:
    before = len(df)
    df = df[~df["discharge_disposition_id"].isin(death_hospice_codes)].copy()
    pre_log["removed_death_hospice_rows"] = int(before - len(df))

# Patient-level leakage control: one encounter per patient.
if {"patient_nbr", "encounter_id"}.issubset(df.columns):
    before = len(df)
    df = df.sort_values("encounter_id").drop_duplicates("patient_nbr", keep="first").copy()
    pre_log["removed_repeat_patient_rows"] = int(before - len(df))

# Near-constant columns.
near_constant = [c for c in df.columns if df[c].nunique(dropna=False) <= 1]
drop_cols = [c for c in ["encounter_id", "patient_nbr", "weight", "payer_code"] if c in df.columns]
drop_cols += [c for c in near_constant if c not in drop_cols]
df = df.drop(columns=drop_cols)
pre_log["dropped_columns"] = drop_cols

# Target is created before dropping readmitted.
df["readmitted_30d"] = (df["readmitted"] == "<30").astype(int)
df = df.drop(columns=["readmitted"])

print(json.dumps(pre_log, indent=2))
print("Post-cohort shape:", df.shape)
print("Target rate:", df["readmitted_30d"].mean())

## 4. Healthcare-specific feature engineering — core novelty component

The raw dataset contains many high-cardinality diagnosis and medication variables. MediShield AI converts them into features that are easier to interpret in a healthcare context.

### Engineered feature families
- **Diagnosis groups:** broad ICD-9 clinical categories from the three diagnosis fields.
- **Diabetes diagnosis count:** number of the three diagnosis positions mapped to diabetes.
- **Medication-regimen summaries:** number of medications going up, down, steady, active, and whether the regimen changed.
- **Prior healthcare utilization:** outpatient + emergency + inpatient visits.
- **Age midpoint:** converts age bands to numeric midpoints.
- **Laboratory missingness:** `NotTested` is retained as information rather than silently imputing a value.

These transformations are part of the proposed framework, not just generic preprocessing.

In [ ]:
# Medical specialty: retain common categories, bucket rare/missing values.
if "medical_specialty" in df.columns:
    top_specialties = df["medical_specialty"].value_counts(dropna=True).nlargest(10).index
    df["medical_specialty"] = df["medical_specialty"].apply(
        lambda x: x if x in top_specialties else ("Missing" if pd.isna(x) else "Other")
    )
if "race" in df.columns:
    df["race"] = df["race"].fillna("Unknown")

# ICD-9 grouping.
def icd9_to_group(code):
    if pd.isna(code): return "Missing"
    s = str(code)
    if s.startswith(("V", "E")): return "Other"
    try: v = float(s)
    except Exception: return "Other"
    if 390 <= v <= 459 or v == 785: return "Circulatory"
    if 460 <= v <= 519 or v == 786: return "Respiratory"
    if 520 <= v <= 579 or v == 787: return "Digestive"
    if 250 <= v < 251: return "Diabetes"
    if 800 <= v <= 999: return "Injury"
    if 710 <= v <= 739: return "Musculoskeletal"
    if 580 <= v <= 629 or v == 788: return "Genitourinary"
    if 140 <= v <= 239: return "Neoplasms"
    if 240 <= v <= 279: return "Endocrine/Metabolic(other)"
    return "Other"

for c in ["diag_1", "diag_2", "diag_3"]:
    if c in df.columns:
        df[c + "_group"] = df[c].apply(icd9_to_group)
        df = df.drop(columns=[c])

diag_group_cols = [c for c in ["diag_1_group", "diag_2_group", "diag_3_group"] if c in df.columns]
if diag_group_cols:
    df["n_diabetes_diag"] = (df[diag_group_cols] == "Diabetes").sum(axis=1)

# Age bands -> midpoints.
age_map = {f"[{a}-{a+10})": a+5 for a in range(0, 90, 10)}
age_map["[90-100)"] = 95
if "age" in df.columns:
    df["age"] = df["age"].map(age_map)

# Medication aggregation.
med_cols = [c for c in ["metformin", "repaglinide", "nateglinide", "chlorpropamide", "glimepiride", "glipizide", "glyburide", "tolbutamide", "pioglitazone", "rosiglitazone", "acarbose", "miglitol", "tolazamide", "insulin", "glyburide-metformin", "glipizide-metformin", "glimepiride-pioglitazone", "metformin-rosiglitazone", "metformin-pioglitazone"] if c in df.columns]
if med_cols:
    df["n_meds_up"] = (df[med_cols] == "Up").sum(axis=1)
    df["n_meds_down"] = (df[med_cols] == "Down").sum(axis=1)
    df["n_meds_steady"] = (df[med_cols] == "Steady").sum(axis=1)
    df["n_meds_active"] = (df[med_cols] != "No").sum(axis=1)
    df["med_regimen_changed"] = ((df["n_meds_up"] + df["n_meds_down"]) > 0).astype(int)
    keep_meds = [c for c in ["insulin", "metformin", "glipizide", "glyburide", "pioglitazone", "rosiglitazone"] if c in med_cols]
    df = df.drop(columns=[c for c in med_cols if c not in keep_meds])

for c in ["max_glu_serum", "A1Cresult"]:
    if c in df.columns:
        df[c] = df[c].fillna("NotTested")

if {"number_outpatient", "number_emergency", "number_inpatient"}.issubset(df.columns):
    df["total_prior_visits"] = df["number_outpatient"] + df["number_emergency"] + df["number_inpatient"]
    df["acute_prior_visits"] = df["number_emergency"] + df["number_inpatient"]

# Remove helper target-independent identifiers if any remain.
df = df.drop(columns=[c for c in ["readmitted_any"] if c in df.columns], errors="ignore")

print("Engineered dataset shape:", df.shape)
print("Target distribution:")
display(df["readmitted_30d"].value_counts().rename_axis("target").reset_index(name="count"))
df.to_csv(DATA_DIR / "medishield_cleaned_features.csv", index=False)
with open(OUT_DIR / "preprocessing_log.json", "w") as f: json.dump(pre_log, f, indent=2)

## 5. Exploratory healthcare analytics

EDA is used to understand the problem and generate hypotheses; it is **not treated as the final project contribution**. The plots below focus on clinically interpretable utilization, age, stay, diagnosis, and medication patterns.

In [ ]:
# Target rate by age band midpoint
plot_df = df.copy()
plot_df["age_band"] = pd.cut(plot_df["age"], bins=[0,30,50,65,80,100], labels=["<30","30–49","50–64","65–79","80+"])

fig, axes = plt.subplots(1, 2, figsize=(13,4.5))
sns.barplot(data=plot_df, x="age_band", y="readmitted_30d", estimator="mean", errorbar=None, ax=axes[0])
axes[0].set_ylabel("30-day readmission rate")
axes[0].set_xlabel("Age group")
axes[0].set_title("30-Day Readmission Rate by Age")

sns.boxplot(data=plot_df, x="readmitted_30d", y="time_in_hospital", ax=axes[1])
axes[1].set_xlabel("30-day readmission (0/1)")
axes[1].set_ylabel("Days in hospital")
axes[1].set_title("Hospital Stay by Readmission Outcome")
plt.tight_layout()
plt.savefig(FIG_DIR / "02_age_and_stay_analysis.png", dpi=180, bbox_inches="tight")
plt.show()

for c in ["total_prior_visits", "acute_prior_visits", "n_meds_active", "n_diabetes_diag"]:
    if c in df.columns:
        summary = df.groupby("readmitted_30d")[c].agg(["mean","median"]).round(2)
        print(f"\n{c}")
        display(summary)

## 6. Leakage-aware train / validation / test design

The data is split **before** SMOTE and preprocessing is fitted only on the training partition. The test set remains untouched until final evaluation.

- Train: 64%
- Validation: 16%
- Test: 20%
- Stratification preserves the target distribution.

Because repeated patients were removed earlier, the final modeling cohort contains one encounter per patient.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from imblearn.over_sampling import SMOTE

X = df.drop(columns=["readmitted_30d"])
y = df["readmitted_30d"].astype(int)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, stratify=y, random_state=RANDOM_STATE)
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.20, stratify=y_train, random_state=RANDOM_STATE)

categorical_cols = X_train.select_dtypes(include=["object", "category"]).columns.tolist()
numeric_cols = [c for c in X_train.columns if c not in categorical_cols]

try:
    ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
except TypeError:
    ohe = OneHotEncoder(handle_unknown="ignore", sparse=False)

preprocessor = ColumnTransformer([
    ("num", StandardScaler(), numeric_cols),
    ("cat", ohe, categorical_cols)
])

X_train_enc = preprocessor.fit_transform(X_train)
X_val_enc = preprocessor.transform(X_val)
X_test_enc = preprocessor.transform(X_test)
feature_names = numeric_cols + list(preprocessor.named_transformers_["cat"].get_feature_names_out(categorical_cols))

smote = SMOTE(random_state=RANDOM_STATE)
X_train_res, y_train_res = smote.fit_resample(X_train_enc, y_train)

print("Train / validation / test:", X_train.shape, X_val.shape, X_test.shape)
print("Positive rate - train:", y_train.mean(), "validation:", y_val.mean(), "test:", y_test.mean())
print("Training counts before SMOTE:", np.bincount(y_train))
print("Training counts after SMOTE:", np.bincount(y_train_res))

joblib.dump(preprocessor, MODEL_DIR / "preprocessor.joblib")

In [ ]:
plt.figure(figsize=(7,4))
counts_before = pd.Series(y_train).value_counts().sort_index()
counts_after = pd.Series(y_train_res).value_counts().sort_index()
ix = np.arange(2)
plt.bar(ix-0.2, counts_before.values, width=0.4, label="Before SMOTE")
plt.bar(ix+0.2, counts_after.values, width=0.4, label="After SMOTE")
plt.xticks(ix, ["No early readmission", "30-day readmission"])
plt.ylabel("Training records")
plt.title("Training-Class Balance Before and After SMOTE")
plt.legend()
plt.tight_layout()
plt.savefig(FIG_DIR / "03_smote_balance.png", dpi=180, bbox_inches="tight")
plt.show()

## 7. Four-model experimental design

MediShield AI deliberately uses **four complementary models**, rather than an arbitrary large list:

1. **Logistic Regression** — transparent linear baseline.
2. **Random Forest** — nonlinear bagging model.
3. **XGBoost** — gradient-boosted tree model.
4. **LightGBM** — efficient gradient-boosted tree model.

Model selection is based primarily on **validation PR-AUC and minority-class F1/recall**, not raw accuracy. Accuracy is retained for reporting because it is required by the project guidelines, but it is not allowed to dominate the selection decision.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
    average_precision_score, confusion_matrix, roc_curve, precision_recall_curve,
    classification_report
)

models = {
    "Logistic Regression": LogisticRegression(max_iter=1500, C=1.0, random_state=RANDOM_STATE),
    "Random Forest": RandomForestClassifier(n_estimators=400, max_depth=14, min_samples_leaf=3, n_jobs=-1, random_state=RANDOM_STATE, class_weight=None),
    "XGBoost": XGBClassifier(n_estimators=350, max_depth=4, learning_rate=0.05, subsample=0.85, colsample_bytree=0.85, eval_metric="logloss", random_state=RANDOM_STATE, n_jobs=-1),
    "LightGBM": LGBMClassifier(n_estimators=350, learning_rate=0.05, num_leaves=31, max_depth=-1, subsample=0.85, colsample_bytree=0.85, random_state=RANDOM_STATE, verbose=-1)
}

def metric_row(y_true, proba, threshold=0.5):
    pred = (proba >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, pred, labels=[0,1]).ravel()
    return {
        "accuracy": accuracy_score(y_true, pred),
        "precision": precision_score(y_true, pred, zero_division=0),
        "recall_sensitivity": recall_score(y_true, pred, zero_division=0),
        "specificity": tn / (tn + fp) if (tn + fp) else 0,
        "f1": f1_score(y_true, pred, zero_division=0),
        "roc_auc": roc_auc_score(y_true, proba),
        "pr_auc": average_precision_score(y_true, proba),
        "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp)
    }

fitted, val_prob, test_prob = {}, {}, {}
rows = []
for name, model in models.items():
    t0 = time.time()
    model.fit(X_train_res, y_train_res)
    pv = model.predict_proba(X_val_enc)[:,1]
    pt = model.predict_proba(X_test_enc)[:,1]
    fitted[name] = model
    val_prob[name] = pv
    test_prob[name] = pt
    m = metric_row(y_test, pt, 0.5)
    m["model"] = name
    m["runtime_sec"] = time.time() - t0
    rows.append(m)

baseline_accuracy = (y_test == 0).mean()
results_default = pd.DataFrame(rows).set_index("model").sort_values(["pr_auc", "f1", "recall_sensitivity"], ascending=False)
print(f"Majority-class accuracy baseline: {baseline_accuracy:.4f}")
display(results_default.round(4))

## 8. Validation-based operating-threshold optimization

The default probability threshold of 0.50 is not assumed to be appropriate for a highly imbalanced healthcare risk problem.

For each candidate model, the operating threshold is selected **only from the validation set** by maximizing validation F1. The untouched test set is then evaluated once at that selected threshold.

This prevents test-set threshold tuning and gives a fair estimate of generalization.

In [ ]:
def best_f1_threshold(y_true, proba):
    precision, recall, thresholds = precision_recall_curve(y_true, proba)
    f1 = 2 * precision * recall / (precision + recall + 1e-12)
    idx = int(np.nanargmax(f1[:-1]))
    return float(thresholds[idx]), float(f1[idx])

threshold_records = []
for name in models:
    thr, val_f1 = best_f1_threshold(y_val, val_prob[name])
    test_m = metric_row(y_test, test_prob[name], thr)
    test_m["model"] = name
    test_m["threshold"] = thr
    test_m["validation_f1"] = val_f1
    threshold_records.append(test_m)

tuned_results = pd.DataFrame(threshold_records).set_index("model").sort_values(["f1", "pr_auc"], ascending=False)
display(tuned_results.round(4))

## 9. MediShield AI ensemble selection and construction

The proposed ensemble is **not selected by test accuracy**. The two strongest candidate models on the validation-oriented criteria are combined using a weighted probability average.

Weights are derived from validation F1 and normalized to sum to one. This keeps the ensemble construction data-driven while preserving a simple, reproducible framework.

In [ ]:
# Rank using validation F1 and validation PR-AUC at the operating threshold.
selection = []
for name in models:
    thr, val_f1 = best_f1_threshold(y_val, val_prob[name])
    val_pr_auc = average_precision_score(y_val, val_prob[name])
    selection.append((name, val_f1, val_pr_auc, thr))
selection_df = pd.DataFrame(selection, columns=["model","val_f1","val_pr_auc","threshold"]).sort_values(["val_f1","val_pr_auc"], ascending=False)
display(selection_df.round(4))

top2 = selection_df.head(2)["model"].tolist()
raw_weights = selection_df.head(2)["val_f1"].to_numpy()
if raw_weights.sum() == 0:
    raw_weights = np.ones(2)
weights = raw_weights / raw_weights.sum()
ensemble_weights = dict(zip(top2, weights))
print("Selected ensemble members:", ensemble_weights)

ens_val_proba = sum(ensemble_weights[n] * val_prob[n] for n in top2)
ens_test_proba = sum(ensemble_weights[n] * test_prob[n] for n in top2)
ens_threshold, ens_val_f1 = best_f1_threshold(y_val, ens_val_proba)
ens_test_metrics = metric_row(y_test, ens_test_proba, ens_threshold)
ens_default_metrics = metric_row(y_test, ens_test_proba, 0.5)
print(f"MediShield validation threshold: {ens_threshold:.4f}")
print("MediShield test metrics at tuned threshold:")
display(pd.DataFrame([ens_test_metrics], index=["MediShield AI"]).round(4))

In [ ]:
# Compare tuned individual models and the proposed ensemble on the untouched test set.
comparison = tuned_results.copy()
comparison["selection_role"] = "Candidate"
comparison.loc[top2, "selection_role"] = "Ensemble member"
comparison.loc["MediShield AI"] = [ens_test_metrics[c] if c in ens_test_metrics else np.nan for c in comparison.columns if c not in ["selection_role"]] + []
# Rebuild a clean comparison table to avoid column-shape ambiguity.
comparison_rows = []
for name in models:
    thr = float(selection_df.loc[selection_df.model == name, "threshold"].iloc[0])
    m = metric_row(y_test, test_prob[name], thr)
    comparison_rows.append({"Model": name, **m, "threshold": thr})
comparison_rows.append({"Model":"MediShield AI", **ens_test_metrics, "threshold":ens_threshold})
comparison_df = pd.DataFrame(comparison_rows).set_index("Model")
print("Final test-set comparison")
display(comparison_df[["accuracy","precision","recall_sensitivity","specificity","f1","roc_auc","pr_auc","threshold"]].round(4))
comparison_df.to_csv(OUT_DIR / "final_model_comparison.csv")

## 10. Performance evaluation

The project reports all metrics requested by the course and adds **PR-AUC**, which is particularly informative for an imbalanced positive class.

- Accuracy
- Precision
- Recall / Sensitivity
- Specificity
- F1-score
- ROC-AUC
- PR-AUC
- Confusion matrix

The final test set is untouched during threshold selection.

In [ ]:
# Final confusion matrix
final_pred = (ens_test_proba >= ens_threshold).astype(int)
cm = confusion_matrix(y_test, final_pred, labels=[0,1])
plt.figure(figsize=(5.5,4.5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False,
            xticklabels=["Predicted No", "Predicted 30-day"],
            yticklabels=["Actual No", "Actual 30-day"])
plt.title("MediShield AI — Final Confusion Matrix")
plt.xlabel("Prediction")
plt.ylabel("Actual")
plt.tight_layout()
plt.savefig(FIG_DIR / "04_final_confusion_matrix.png", dpi=200, bbox_inches="tight")
plt.show()

# ROC and PR curves
fig, ax = plt.subplots(figsize=(7,5.5))
for name in models:
    fpr, tpr, _ = roc_curve(y_test, test_prob[name])
    ax.plot(fpr, tpr, label=f"{name} (AUC={roc_auc_score(y_test,test_prob[name]):.3f})")
fpr, tpr, _ = roc_curve(y_test, ens_test_proba)
ax.plot(fpr, tpr, linewidth=2.5, label=f"MediShield AI (AUC={roc_auc_score(y_test,ens_test_proba):.3f})")
ax.plot([0,1],[0,1],"k--",linewidth=1)
ax.set_xlabel("False Positive Rate"); ax.set_ylabel("True Positive Rate")
ax.set_title("ROC Curve Comparison")
ax.legend(fontsize=8)
plt.tight_layout(); plt.savefig(FIG_DIR / "05_roc_comparison.png", dpi=200, bbox_inches="tight"); plt.show()

fig, ax = plt.subplots(figsize=(7,5.5))
for name in models:
    p, r, _ = precision_recall_curve(y_test, test_prob[name])
    ax.plot(r, p, label=f"{name} (AP={average_precision_score(y_test,test_prob[name]):.3f})")
p, r, _ = precision_recall_curve(y_test, ens_test_proba)
ax.plot(r, p, linewidth=2.5, label=f"MediShield AI (AP={average_precision_score(y_test,ens_test_proba):.3f})")
ax.axhline(y_test.mean(), linestyle="--", linewidth=1, label="Positive-class prevalence")
ax.set_xlabel("Recall / Sensitivity"); ax.set_ylabel("Precision")
ax.set_title("Precision–Recall Curve Comparison")
ax.legend(fontsize=8)
plt.tight_layout(); plt.savefig(FIG_DIR / "06_precision_recall_comparison.png", dpi=200, bbox_inches="tight"); plt.show()

In [ ]:
# Metric comparison for the final thresholded results.
metric_cols = ["accuracy","precision","recall_sensitivity","specificity","f1","roc_auc","pr_auc"]
ax = comparison_df[metric_cols].plot(kind="bar", figsize=(12,5.5))
ax.set_ylabel("Score")
ax.set_title("Final Test-Set Performance Comparison")
ax.set_ylim(0,1)
plt.xticks(rotation=25, ha="right")
plt.tight_layout(); plt.savefig(FIG_DIR / "07_final_metric_comparison.png", dpi=200, bbox_inches="tight"); plt.show()

# Threshold impact for MediShield AI
threshold_grid = np.linspace(max(0.01, ens_threshold-0.25), min(0.99, ens_threshold+0.25), 80)
curve = pd.DataFrame([{"threshold":t, **metric_row(y_test, ens_test_proba, t)} for t in threshold_grid])
plt.figure(figsize=(8,5))
for c in ["precision","recall_sensitivity","f1","accuracy"]:
    plt.plot(curve["threshold"], curve[c], label=c)
plt.axvline(ens_threshold, linestyle="--", label=f"Selected={ens_threshold:.3f}")
plt.xlabel("Probability threshold"); plt.ylabel("Score"); plt.title("MediShield AI Threshold Trade-off")
plt.legend(); plt.tight_layout(); plt.savefig(FIG_DIR / "08_threshold_tradeoff.png", dpi=200, bbox_inches="tight"); plt.show()

## 11. Explainable AI — SHAP

The goal of SHAP is not merely to produce a feature-importance chart. MediShield AI uses SHAP to answer two questions:

1. **Global:** Which factors most influence the ensemble's risk estimates across patients?
2. **Individual:** Why was this particular patient assigned a high or low risk?

For the selected ensemble members, SHAP values are calculated on a common test sample and combined using the same ensemble weights. This keeps the explanation aligned with the final ensemble rather than explaining an unrelated model.

In [ ]:
import shap

rng = np.random.RandomState(RANDOM_STATE)
sample_n = min(500, X_test_enc.shape[0])
sample_idx = rng.choice(X_test_enc.shape[0], size=sample_n, replace=False)
X_shap = X_test_enc[sample_idx]

shap_by_model = {}
for name in top2:
    model = fitted[name]
    # TreeExplainer is used for tree-based models. If the selected model is not tree-based,
    # use a model-agnostic KernelExplainer on a compact background sample.
    if name in ["Random Forest", "XGBoost", "LightGBM"]:
        explainer = shap.TreeExplainer(model)
        sv = explainer.shap_values(X_shap)
        if isinstance(sv, list): sv = sv[1]
        sv = np.asarray(sv)
        if sv.ndim == 3: sv = sv[:,:,1]
    else:
        background = shap.sample(X_train_enc, min(80, X_train_enc.shape[0]), random_state=RANDOM_STATE)
        explainer = shap.KernelExplainer(model.predict_proba, background)
        sv = explainer.shap_values(X_shap, nsamples=100)
        if isinstance(sv, list): sv = sv[1]
        sv = np.asarray(sv)
    shap_by_model[name] = sv

ensemble_shap = sum(ensemble_weights[n] * shap_by_model[n] for n in top2)
mean_abs = np.abs(ensemble_shap).mean(axis=0)
order = np.argsort(mean_abs)[::-1]
shap_importance = pd.DataFrame({"feature": np.array(feature_names)[order], "mean_abs_shap": mean_abs[order]})
display(shap_importance.head(20).round(5))
shap_importance.to_csv(OUT_DIR / "shap_global_importance.csv", index=False)

plt.figure(figsize=(9,7))
shap.summary_plot(ensemble_shap, X_shap, feature_names=feature_names, max_display=20, show=False)
plt.title("MediShield AI — Global SHAP Summary")
plt.tight_layout(); plt.savefig(FIG_DIR / "09_shap_global_summary.png", dpi=200, bbox_inches="tight"); plt.show()

In [ ]:
# SHAP bar plot for report/PPT
plt.figure(figsize=(9,6))
show = shap_importance.head(15).sort_values("mean_abs_shap")
plt.barh(show["feature"], show["mean_abs_shap"])
plt.xlabel("Mean absolute SHAP value")
plt.title("Top 15 Global Contributors to MediShield AI Risk")
plt.tight_layout(); plt.savefig(FIG_DIR / "10_shap_global_bar.png", dpi=200, bbox_inches="tight"); plt.show()

# Individual explanation: choose the highest-risk test patient under the final model.
high_idx = int(np.argmax(ens_test_proba))
patient_global_index = X_test.index[high_idx]
patient_x = X_test_enc[high_idx:high_idx+1]
patient_probability = float(ens_test_proba[high_idx])
patient_actual = int(y_test.iloc[high_idx])
patient_shap = ensemble_shap[np.where(sample_idx == high_idx)[0][0]] if high_idx in sample_idx else None

print(f"Selected high-risk test patient index: {patient_global_index}")
print(f"Predicted 30-day readmission probability: {patient_probability:.3f}")
print(f"Actual outcome: {patient_actual}")

if patient_shap is not None:
    local = pd.DataFrame({"feature": feature_names, "shap_value": patient_shap})
    local["abs_shap"] = local["shap_value"].abs()
    display(local.sort_values("abs_shap", ascending=False).head(15).round(5))
    # Waterfall using an additive SHAP Explanation object.
    base = float(np.mean([np.mean(s) for s in shap_by_model.values()]))
    explanation = shap.Explanation(values=patient_shap, base_values=base, data=patient_x[0], feature_names=feature_names)
    shap.plots.waterfall(explanation, max_display=15, show=False)
    plt.title("MediShield AI — Individual High-Risk Patient Explanation")
    plt.tight_layout(); plt.savefig(FIG_DIR / "11_shap_individual_waterfall.png", dpi=200, bbox_inches="tight"); plt.show()

## 12. Patient risk stratification

The model produces a probability rather than only a class label. For reporting, patients are placed into three project-defined risk bands using the tuned operating threshold as the high-risk anchor.

- **Low:** probability below 0.50 × selected threshold
- **Moderate:** probability from 0.50 × selected threshold up to the selected threshold
- **High:** probability at or above the selected threshold

These are **analytical risk bands for this project**, not clinical guidelines. They should not be presented as validated clinical decision thresholds.

In [ ]:
def risk_band(p):
    if p >= ens_threshold: return "High"
    if p >= 0.5 * ens_threshold: return "Moderate"
    return "Low"

risk_output = X_test.copy()
risk_output["actual_readmitted_30d"] = y_test.values
risk_output["predicted_probability"] = ens_test_proba
risk_output["predicted_class"] = final_pred
risk_output["risk_band"] = [risk_band(p) for p in ens_test_proba]

risk_summary = risk_output.groupby("risk_band").agg(
    patients=("risk_band","size"),
    mean_predicted_risk=("predicted_probability","mean"),
    observed_30d_rate=("actual_readmitted_30d","mean")
).reindex(["Low","Moderate","High"])
display(risk_summary.round(4))
risk_summary.to_csv(OUT_DIR / "risk_band_summary.csv")
risk_output.to_csv(OUT_DIR / "test_patient_risk_scores.csv", index=False)

plt.figure(figsize=(7,4.5))
sns.barplot(x=risk_summary.index, y=risk_summary["observed_30d_rate"].values, errorbar=None)
plt.ylabel("Observed 30-day readmission rate")
plt.xlabel("MediShield AI risk band")
plt.title("Observed Readmission Rate Across Project Risk Bands")
plt.tight_layout(); plt.savefig(FIG_DIR / "12_risk_band_observed_rate.png", dpi=200, bbox_inches="tight"); plt.show()

## 13. Healthcare insights generated from the data

The following tables are designed to support the report's healthcare-insight section. They deliberately show **observed group differences**, rather than asserting causation.

The model's SHAP results should be discussed as **predictive associations/contributions**, not causal effects.

In [ ]:
# Group-level observed rates for key healthcare variables.
insight_tables = {}

if "n_meds_active" in df.columns:
    temp = df.copy()
    temp["medication_burden"] = pd.qcut(temp["n_meds_active"].rank(method="first"), 4, labels=["Q1","Q2","Q3","Q4"])
    insight_tables["Medication burden"] = temp.groupby("medication_burden", observed=True)["readmitted_30d"].agg(["count","mean"]).rename(columns={"mean":"30d_readmission_rate"})

if "total_prior_visits" in df.columns:
    temp = df.copy()
    temp["prior_utilization"] = pd.cut(temp["total_prior_visits"], bins=[-1,0,2,5,10,np.inf], labels=["0","1–2","3–5","6–10","11+"])
    insight_tables["Prior utilization"] = temp.groupby("prior_utilization", observed=True)["readmitted_30d"].agg(["count","mean"]).rename(columns={"mean":"30d_readmission_rate"})

if "time_in_hospital" in df.columns:
    temp = df.copy()
    temp["stay_group"] = pd.cut(temp["time_in_hospital"], bins=[0,2,4,7,14,np.inf], labels=["1–2","3–4","5–7","8–14","15+"])
    insight_tables["Hospital stay"] = temp.groupby("stay_group", observed=True)["readmitted_30d"].agg(["count","mean"]).rename(columns={"mean":"30d_readmission_rate"})

for title, table in insight_tables.items():
    print("\n", title)
    display((table*100).round(2) if "30d_readmission_rate" in table.columns else table)

# Top SHAP factors as a concise report table.
print("\nTop global predictive contributors")
display(shap_importance.head(10).round(5))

## 14. Robustness checks and project limitations

### Robustness checks
- The operating threshold is selected on validation data, not test data.
- SMOTE is applied only to training data.
- The test set remains untouched during model fitting and threshold selection.
- A majority-class baseline is reported so high accuracy is not mistaken for useful minority-class detection.
- PR-AUC and recall are reported alongside accuracy.

### Limitations
- The dataset is historical (1999–2008) and comes from U.S. hospital encounters.
- Removing repeat encounters reduces leakage risk but also removes longitudinal information.
- Missingness is substantial for some fields; therefore, some potentially informative variables are excluded or grouped.
- The framework has not been externally or prospectively validated.
- Risk bands are project-defined analytical categories, not clinical guidelines.
- Predictive associations should not be interpreted as causal relationships.
- The model should be considered a research/analytics prototype rather than a clinical decision system.

## 15. Final project checklist

This notebook now provides the implementation evidence for the course requirements:

| Course requirement | Notebook evidence |
|---|---|
| Dataset description | Dataset audit + target definition |
| Problem identification | 30-day readmission objective |
| Novel/improved approach | Healthcare feature engineering + imbalance-aware ensemble + threshold optimization + SHAP |
| Preprocessing | Missing-value handling, identifiers, diagnosis grouping, encoding, scaling |
| Class imbalance | SMOTE on training only |
| ML | Logistic Regression, Random Forest, XGBoost, LightGBM |
| Ensemble learning | Weighted top-2 validation-selected ensemble |
| Evaluation | Accuracy, precision, recall, specificity, F1, ROC-AUC, PR-AUC, confusion matrix |
| Visualization | EDA, imbalance, ROC, PR, metric comparison, threshold trade-off, SHAP, risk bands |
| Explainable AI | Global SHAP + individual patient SHAP |
| Healthcare insights | Utilization, medication burden, hospital stay, predictive contributors |
| Reproducibility | Fixed random seed, saved preprocessing/model/results |

**Still required outside this notebook:** literature review, properly cited references, final written report, GitHub repository, and presentation. A Power BI/Tableau dashboard is optional unless your instructor separately requires it.

## 16. Save reproducibility package and final outputs

In [ ]:
# Save final model components and a machine-readable project summary.
joblib.dump({"models": fitted, "ensemble_members": top2, "ensemble_weights": ensemble_weights, "threshold": ens_threshold}, MODEL_DIR / "medishield_ai_models.joblib")

project_summary = {
    "project_title": "MediShield AI: Explainable AI Framework for Early Prediction of 30-Day Patient Readmission Risk",
    "task": "Binary classification of 30-day readmission",
    "positive_class": "readmitted within 30 days",
    "raw_shape": list(df_raw.shape),
    "modeling_shape": list(df.shape),
    "train_rows": int(len(X_train)), "validation_rows": int(len(X_val)), "test_rows": int(len(X_test)),
    "candidate_models": list(models.keys()),
    "ensemble_members": top2,
    "ensemble_weights": {k: float(v) for k,v in ensemble_weights.items()},
    "threshold": float(ens_threshold),
    "majority_class_accuracy": float(baseline_accuracy),
    "final_test_metrics": {k: float(v) for k,v in ens_test_metrics.items() if isinstance(v,(int,float,np.integer,np.floating))},
    "novelty": [
        "Healthcare-specific diagnosis, medication and utilization feature engineering",
        "Patient-level leakage control",
        "Training-only SMOTE",
        "Validation-driven multi-metric ensemble selection",
        "Validation-only threshold optimization",
        "Ensemble-consistent SHAP explanations",
        "Analytical patient risk stratification"
    ]
}
with open(OUT_DIR / "medishield_project_summary.json", "w") as f:
    json.dump(project_summary, f, indent=2)

print("Saved project package:")
for p in [DATA_DIR, FIG_DIR, OUT_DIR, MODEL_DIR]:
    print(" -", p.resolve())
print("\nFinal MediShield AI test metrics:")
print(pd.Series(ens_test_metrics).round(4))

# End of MediShield AI implementation

**Interpretation rule for the final report:** report the actual values generated above. Do not replace them with target accuracies or values from online implementations. In particular, if accuracy is high while minority-class recall/F1 is modest, explain the class-imbalance trade-off rather than presenting accuracy alone.